# 🔥 Punjab Stubble Fire Prediction — Complete EDA & Feature Engineering
**AI3011 · Machine Learning & Pattern Recognition · Plaksha University**  
**Dataset:** NASA FIRMS — MODIS C61 + VIIRS NOAA-20 C2 · Punjab · Oct–Nov · 2018–2023  

---
### Notebook Structure
1. Data Loading & Merging
2. Data Cleaning & Confidence Filtering
3. Geographic Filtering — Punjab Bounding Box
4. Seasonal Filtering — Oct–Nov Only
5. EDA — Fires by Year
6. EDA — The 4 PM Evasion Shift
7. EDA — Spatial Distribution & Grid Heatmap
8. EDA — Weekly Fire Pattern
9. Feature Engineering — Building the ML Input Table
10. Feature Correlation Analysis
11. Final Dataset Summary
---

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

print('All imports successful ✅')

---
## 1. Data Loading & Merging
We use two NASA FIRMS sensors:
- **MODIS C61** — 1km resolution, longer historical record
- **VIIRS NOAA-20 C2** — 375m resolution, newer & more precise  

Both are downloaded from firms.modaps.eosdis.nasa.gov for the Punjab bounding box, 2018–2023.

In [ ]:
# ── Load raw CSVs ────────────────────────────────────────────────
modis    = pd.read_csv('fire_archive_M-C61_720765.csv')
viirs_n20 = pd.read_csv('fire_archive_J1V-C2_720766.csv')

print(f'MODIS raw shape:     {modis.shape}')
print(f'VIIRS N20 raw shape: {viirs_n20.shape}')
print(f'\nMODIS columns:  {modis.columns.tolist()}')
print(f'VIIRS columns:  {viirs_n20.columns.tolist()}')

In [ ]:
# ── Keep common columns ──────────────────────────────────────────
COMMON_COLS = ['latitude', 'longitude', 'brightness', 'acq_date',
               'acq_time', 'satellite', 'instrument', 'confidence', 'frp', 'daynight']

modis     = modis[COMMON_COLS].copy()
viirs_n20 = viirs_n20[COMMON_COLS].copy()

# ── Normalise VIIRS confidence (l/n/h → 30/60/90) ───────────────
# MODIS uses numeric 0-100; VIIRS uses categorical low/nominal/high
viirs_map = {'l': 30, 'n': 60, 'h': 90}
viirs_n20['confidence'] = (
    viirs_n20['confidence'].astype(str).str.lower().map(viirs_map)
)
print('VIIRS confidence unique values after mapping:', viirs_n20['confidence'].unique())

# ── Tag sensor ──────────────────────────────────────────────────
modis['sensor']     = 'MODIS'
viirs_n20['sensor'] = 'VIIRS'

# ── Merge ───────────────────────────────────────────────────────
fires = pd.concat([modis, viirs_n20], ignore_index=True)

# ── Parse dates & times ─────────────────────────────────────────
fires['acq_date'] = pd.to_datetime(fires['acq_date'])
fires['acq_time'] = fires['acq_time'].astype(str).str.zfill(4)
fires['hour_utc'] = fires['acq_time'].str[:2].astype(int)
fires['hour_ist'] = (fires['hour_utc'] + 5) % 24   # IST = UTC+5:30 (approx)
fires['year']     = fires['acq_date'].dt.year
fires['month']    = fires['acq_date'].dt.month
fires['week']     = fires['acq_date'].dt.isocalendar().week.astype(int)

# ── Drop duplicates ──────────────────────────────────────────────
before_dedup = len(fires)
fires = fires.drop_duplicates(subset=['latitude', 'longitude', 'acq_date', 'acq_time'])
print(f'\nAfter deduplication: {before_dedup:,} → {len(fires):,} rows')
print(f'Sensor breakdown:\n{fires["sensor"].value_counts()}')

---
## 2. Data Cleaning — Confidence Filter
Low-confidence detections are likely false positives (hot roads, industrial sites, reflected sunlight).  
We drop them to keep only reliable fire detections.

In [ ]:
# ── Visualise confidence BEFORE filtering ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# MODIS
modis_conf = fires[fires['sensor'] == 'MODIS']['confidence']
axes[0].hist(modis_conf, bins=25, color='#4A90D9', edgecolor='white')
axes[0].axvline(x=50, color='red', linestyle='--', linewidth=2, label='Threshold = 50')
axes[0].set_title('MODIS Confidence Distribution', fontweight='bold')
axes[0].set_xlabel('Confidence Score (0–100)')
axes[0].set_ylabel('Count')
axes[0].legend()
drop_modis = (modis_conf < 50).sum()
axes[0].annotate(f'{drop_modis:,} rows will be dropped',
                 xy=(0.05, 0.93), xycoords='axes fraction', fontsize=9, color='red')

# VIIRS
viirs_conf = fires[fires['sensor'] == 'VIIRS']['confidence'].value_counts().sort_index()
bar_colors = ['#E8512A', '#4A90D9', '#2ECC71']
bars = axes[1].bar(viirs_conf.index, viirs_conf.values, color=bar_colors, width=8, edgecolor='white')
axes[1].set_xticks([30, 60, 90])
axes[1].set_xticklabels(['Low (30)\n← DROP', 'Nominal (60)\nKEEP', 'High (90)\nKEEP'])
axes[1].set_title('VIIRS Confidence Categories', fontweight='bold')
axes[1].set_ylabel('Count')
for bar, val in zip(bars, viirs_conf.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
                 f'{val:,}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Confidence Score Distributions — Before Filtering', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_confidence_distributions.png', bbox_inches='tight')
plt.show()

# ── Apply filter ────────────────────────────────────────────────
before = len(fires)
fires = fires[fires['confidence'] >= 50].copy()
after  = len(fires)
print(f'Before confidence filter: {before:,}')
print(f'After  confidence filter: {after:,}')
print(f'Dropped: {before - after:,} ({(before-after)/before*100:.1f}%) low-confidence detections')

---
## 3. Geographic Filtering — Punjab Bounding Box

In [ ]:
# ── Plot India first to show context ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(fires['longitude'], fires['latitude'], s=0.5, alpha=0.2, color='#E8512A')
axes[0].set_title('All India Fire Detections (Raw)', fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

# ── Apply Punjab bounding box ────────────────────────────────────
LAT_MIN, LAT_MAX = 29.7, 32.5
LON_MIN, LON_MAX = 74.0, 76.5

punjab = fires[
    (fires['latitude']  >= LAT_MIN) & (fires['latitude']  <= LAT_MAX) &
    (fires['longitude'] >= LON_MIN) & (fires['longitude'] <= LON_MAX)
].copy()

axes[1].scatter(punjab['longitude'], punjab['latitude'], s=1, alpha=0.3, color='#E8512A')
axes[1].set_xlim(LON_MIN, LON_MAX)
axes[1].set_ylim(LAT_MIN, LAT_MAX)
axes[1].set_title('Punjab Bounding Box (Zoomed)', fontweight='bold')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.suptitle('Geographic Filter: All India → Punjab', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_geographic_filter.png', bbox_inches='tight')
plt.show()

print(f'Before Punjab filter: {len(fires):,}')
print(f'After  Punjab filter: {len(punjab):,}')

---
## 4. Seasonal Filtering — Oct–Nov Stubble Burning Season

In [ ]:
# ── Show monthly distribution before filtering ──────────────────
monthly = punjab.groupby('month').size().reset_index(name='count')
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly['month_name'] = monthly['month'].map(month_names)

fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#E8512A' if m in [10, 11] else '#B0C4DE' for m in monthly['month']]
bars = ax.bar(monthly['month_name'], monthly['count'], color=colors, edgecolor='white')
ax.set_title('Fire Detections by Month — Punjab (All Years)', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Fire Detections')

legend_elements = [
    mpatches.Patch(facecolor='#E8512A', label='Oct–Nov: Stubble burning season (KEEP)'),
    mpatches.Patch(facecolor='#B0C4DE', label='Other months (DROP)')
]
ax.legend(handles=legend_elements)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig3_monthly_distribution.png', bbox_inches='tight')
plt.show()

# ── Apply seasonal filter ────────────────────────────────────────
before = len(punjab)
punjab = punjab[punjab['month'].isin([10, 11])].copy()
print(f'Before seasonal filter: {before:,}')
print(f'After  seasonal filter: {len(punjab):,}')
print(f'\nYears in dataset: {sorted(punjab["year"].unique())}')
print(f'Sensor breakdown:\n{punjab["sensor"].value_counts()}')

---
## 5. EDA — Fires by Year
How has stubble burning changed over time?

In [ ]:
yearly = punjab.groupby('year').size().reset_index(name='fire_count')
sensor_year = punjab.groupby(['year', 'sensor']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(yearly['year'], yearly['fire_count'],
              color='#E8512A', edgecolor='white', linewidth=0.8, width=0.6)

for bar, (_, row) in zip(bars, yearly.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{row['fire_count']:,}", ha='center', va='bottom', fontsize=10, fontweight='bold')
    # Sensor breakdown inside bar
    yr = row['year']
    m_count = sensor_year.loc[yr, 'MODIS']  if 'MODIS'  in sensor_year.columns else 0
    v_count = sensor_year.loc[yr, 'VIIRS']  if 'VIIRS'  in sensor_year.columns else 0
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.45,
            f'M:{m_count:,}\nV:{v_count:,}', ha='center', fontsize=7.5,
            color='white', alpha=0.9)

ax.set_title('Punjab Stubble Fires by Year — Oct/Nov Season\n(MODIS + VIIRS, Confidence ≥ 50%)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Fire Detections', fontsize=12)
ax.set_xticks(yearly['year'])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig4_fires_by_year.png', bbox_inches='tight')
plt.show()

---
## 6. EDA — The 4 PM Evasion Shift
**Key finding from Jethva et al. (NASA, 2025):** Farmers learned the satellite overpass time (~1:30 PM IST) and shifted burning to after 4 PM to avoid detection. We verify this directly in our data.

In [ ]:
# ── Overall hourly distribution (IST) ───────────────────────────
hour_counts_ist = punjab['hour_ist'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: overall
colors = ['#E8512A' if h >= 16 else '#4A90D9' if h in [13, 14] else '#B0C4DE'
          for h in hour_counts_ist.index]
axes[0].bar(hour_counts_ist.index, hour_counts_ist.values, color=colors, edgecolor='white')
axes[0].axvline(x=13.5, color='#2C5F8A', linestyle='--', linewidth=2)
axes[0].text(13.7, hour_counts_ist.max() * 0.88,
             'Satellite\noverpass\n~1:30 PM', fontsize=8, color='#2C5F8A')
axes[0].axvspan(16, 23, alpha=0.08, color='#E8512A')

after4_pct = (punjab['hour_ist'] >= 16).mean() * 100
axes[0].annotate(f'{after4_pct:.1f}% of fires after 4 PM IST',
                 xy=(0.97, 0.95), xycoords='axes fraction', ha='right',
                 fontsize=10, fontweight='bold', color='#E8512A')

legend_els = [
    mpatches.Patch(facecolor='#4A90D9', label='Overpass window'),
    mpatches.Patch(facecolor='#E8512A', label='Post-4PM evasion'),
    mpatches.Patch(facecolor='#B0C4DE', label='Other hours')
]
axes[0].legend(handles=legend_els, fontsize=9)
axes[0].set_title('Fire Detections by Hour (IST) — All Data', fontweight='bold')
axes[0].set_xlabel('Hour of Day (IST)')
axes[0].set_ylabel('Fire Detections')
axes[0].set_xticks(range(0, 24))

# Right: MODIS vs VIIRS side by side — does VIIRS catch more evening fires?
modis_hours = punjab[punjab['sensor'] == 'MODIS']['hour_ist'].value_counts().sort_index()
viirs_hours = punjab[punjab['sensor'] == 'VIIRS']['hour_ist'].value_counts().sort_index()

all_hours = sorted(set(modis_hours.index) | set(viirs_hours.index))
modis_vals = [modis_hours.get(h, 0) for h in all_hours]
viirs_vals = [viirs_hours.get(h, 0) for h in all_hours]

x = np.arange(len(all_hours))
w = 0.4
axes[1].bar(x - w/2, modis_vals, width=w, label='MODIS', color='#4A90D9', edgecolor='white')
axes[1].bar(x + w/2, viirs_vals, width=w, label='VIIRS N20', color='#E8512A', edgecolor='white')
axes[1].axvline(x=all_hours.index(13) if 13 in all_hours else 13,
                color='black', linestyle='--', linewidth=1.5, alpha=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(all_hours, fontsize=8)
axes[1].set_title('MODIS vs VIIRS — Hourly Breakdown\n(Does VIIRS catch more evening fires?)',
                   fontweight='bold')
axes[1].set_xlabel('Hour of Day (IST)')
axes[1].set_ylabel('Fire Detections')
axes[1].legend()

plt.suptitle('The 4 PM Evasion Shift — Farmers Dodge Satellite Detection', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig5_4pm_shift.png', bbox_inches='tight')
plt.show()

# Mean hour by sensor
print('Mean detection hour (IST) by sensor:')
print(punjab.groupby('sensor')['hour_ist'].mean().round(2))

---
## 7. EDA — Spatial Distribution & Grid Heatmap
Which parts of Punjab burn the most? We divide Punjab into 7km × 7km grid cells.

In [ ]:
# ── Build grid ───────────────────────────────────────────────────
GRID_DEG = 0.07   # ~7km in degrees latitude

punjab['grid_x'] = ((punjab['longitude'] - LON_MIN) / GRID_DEG).astype(int)
punjab['grid_y'] = ((punjab['latitude']  - LAT_MIN) / GRID_DEG).astype(int)
punjab['grid_id'] = punjab['grid_x'].astype(str) + '_' + punjab['grid_y'].astype(str)

print(f'Total unique 7km grid cells with fire activity: {punjab["grid_id"].nunique()}')

# ── Grid-level fire counts ────────────────────────────────────────
grid_totals = punjab.groupby(['grid_x', 'grid_y']).agg(
    fire_count  = ('acq_date', 'count'),
    avg_frp     = ('frp', 'mean'),
).reset_index()

# Convert back to lat/lon for plotting
grid_totals['lon_center'] = LON_MIN + (grid_totals['grid_x'] + 0.5) * GRID_DEG
grid_totals['lat_center'] = LAT_MIN + (grid_totals['grid_y'] + 0.5) * GRID_DEG

# ── Plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: raw scatter coloured by FRP intensity
sc = axes[0].scatter(punjab['longitude'], punjab['latitude'],
                     c=punjab['frp'], cmap='hot_r', s=1, alpha=0.3,
                     vmin=0, vmax=punjab['frp'].quantile(0.95))
plt.colorbar(sc, ax=axes[0], label='Fire Radiative Power (MW)')
axes[0].set_title('Raw Fire Detections — FRP Intensity', fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_xlim(LON_MIN, LON_MAX)
axes[0].set_ylim(LAT_MIN, LAT_MAX)

# Right: grid heatmap
sc2 = axes[1].scatter(grid_totals['lon_center'], grid_totals['lat_center'],
                      c=grid_totals['fire_count'], cmap='YlOrRd',
                      s=60, marker='s', alpha=0.85,
                      vmin=0, vmax=grid_totals['fire_count'].quantile(0.95))
plt.colorbar(sc2, ax=axes[1], label='Total Fire Events (all years)')
axes[1].set_title(f'7km × 7km Grid Fire Density\n({punjab["grid_id"].nunique()} active cells)',
                   fontweight='bold')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_xlim(LON_MIN, LON_MAX)
axes[1].set_ylim(LAT_MIN, LAT_MAX)

plt.suptitle('Punjab Spatial Fire Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_spatial_heatmap.png', bbox_inches='tight')
plt.show()

---
## 8. EDA — Weekly Fire Pattern
When within Oct–Nov does burning peak? Critical for prediction timing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: weekly activity across all years
weekly_all = punjab.groupby('week').size().reset_index(name='count')
axes[0].fill_between(weekly_all['week'], weekly_all['count'], alpha=0.3, color='#E8512A')
axes[0].plot(weekly_all['week'], weekly_all['count'], color='#E8512A', linewidth=2.5)
peak_week = weekly_all.loc[weekly_all['count'].idxmax(), 'week']
axes[0].axvline(x=peak_week, color='black', linestyle='--', linewidth=1.5)
axes[0].text(peak_week + 0.2, weekly_all['count'].max() * 0.92,
             f'Peak: Week {peak_week}', fontsize=10, fontweight='bold')
axes[0].set_title('Total Weekly Fire Activity (All Years)', fontweight='bold')
axes[0].set_xlabel('ISO Week Number')
axes[0].set_ylabel('Fire Detections')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Right: year-over-year lines
palette = sns.color_palette('tab10', n_colors=len(punjab['year'].unique()))
for i, yr in enumerate(sorted(punjab['year'].unique())):
    yw = punjab[punjab['year'] == yr].groupby('week').size().reset_index(name='count')
    axes[1].plot(yw['week'], yw['count'], marker='o', linewidth=2,
                 color=palette[i], label=str(yr), alpha=0.85, markersize=4)

axes[1].set_title('Weekly Fire Activity — Year by Year', fontweight='bold')
axes[1].set_xlabel('ISO Week Number')
axes[1].set_ylabel('Fire Detections')
axes[1].legend(title='Year', fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Temporal Fire Patterns — When Does Punjab Burn?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_weekly_patterns.png', bbox_inches='tight')
plt.show()

---
## 9. Feature Engineering — Building the ML Input Table
We aggregate to **grid × week** level and engineer 8 features that will train XGBoost.

| Feature | Type | Description |
|---|---|---|
| `fire_count` | **Target** | How many fires in this grid cell this week |
| `fire_count_last_week` | Lag | Fires in same cell last week |
| `same_week_last_year` | Lag | Fires in same cell, same week, last year |
| `3yr_avg` | Rolling | 3-year average for same cell + week |
| `neighbor_fire_count` | Spatial | Sum of fires in 8 surrounding cells |
| `avg_frp` | Intensity | Average fire radiative power |
| `avg_brightness` | Intensity | Average brightness temperature |
| `night_fire_pct` | Evasion | % of fires detected at night |
| `week_of_season` | Temporal | Week 1–9 within Oct–Nov season |

In [ ]:
# ── Base grid × week aggregation ────────────────────────────────
grid_week = (
    punjab.groupby(['grid_id', 'grid_x', 'grid_y', 'year', 'week'])
    .agg(
        fire_count     = ('acq_date',   'count'),
        avg_frp        = ('frp',        'mean'),
        avg_brightness = ('brightness', 'mean'),
        avg_confidence = ('confidence', 'mean'),
        night_fire_pct = ('daynight',   lambda x: (x == 'N').mean()),
    )
    .reset_index()
)

grid_week = grid_week.sort_values(['grid_id', 'year', 'week']).reset_index(drop=True)
print(f'Base grid × week table: {grid_week.shape}')
print(f'Unique grid cells: {grid_week["grid_id"].nunique()}')
grid_week.head(5)

In [ ]:
# ── Feature 1: fire_count_last_week ─────────────────────────────
grid_week['fire_count_last_week'] = (
    grid_week.groupby(['grid_id', 'year'])['fire_count']
    .shift(1).fillna(0)
)

# ── Feature 2: same_week_last_year ──────────────────────────────
prev_year = grid_week[['grid_id', 'year', 'week', 'fire_count']].copy()
prev_year['year'] = prev_year['year'] + 1
prev_year = prev_year.rename(columns={'fire_count': 'same_week_last_year'})
grid_week = grid_week.merge(prev_year, on=['grid_id', 'year', 'week'], how='left')
grid_week['same_week_last_year'] = grid_week['same_week_last_year'].fillna(0)

# ── Feature 3: 3-year rolling average (same week) ───────────────
def add_3yr_avg(group):
    group = group.sort_values('year')
    group['3yr_avg'] = (
        group['fire_count'].shift(1).rolling(3, min_periods=1).mean().fillna(0)
    )
    return group

grid_week = (
    grid_week.groupby(['grid_id', 'week'], group_keys=False)
    .apply(add_3yr_avg)
    .reset_index(drop=True)
)

# ── Feature 4: neighbor_fire_count ──────────────────────────────
print('Computing neighbor fire counts (may take ~1 min)...')
lookup = grid_week.set_index(['grid_x', 'grid_y', 'year', 'week'])['fire_count'].to_dict()

def neighbor_count(row):
    total = 0
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue
            total += lookup.get((row['grid_x']+dx, row['grid_y']+dy, row['year'], row['week']), 0)
    return total

grid_week['neighbor_fire_count'] = grid_week.apply(neighbor_count, axis=1)

# ── Feature 5: week_of_season ────────────────────────────────────
# Week 40 = season week 1, week 48 = season week 9
grid_week['week_of_season'] = (grid_week['week'] - 39).clip(1, 9)

print('✅ All features engineered!')
print(f'Final dataset shape: {grid_week.shape}')
print(f'Features: {list(grid_week.columns)}')

In [ ]:
# ── Show the feature table ───────────────────────────────────────
feature_cols = [
    'grid_id', 'year', 'week_of_season',
    'fire_count',            # TARGET
    'fire_count_last_week',  # Feature 1
    'same_week_last_year',   # Feature 2
    '3yr_avg',               # Feature 3
    'neighbor_fire_count',   # Feature 4
    'avg_frp',               # Feature 5
    'avg_brightness',        # Feature 6
    'night_fire_pct',        # Feature 7
    'avg_confidence',        # Feature 8
]

print('=== ML FEATURE TABLE — Sample (rows with fire activity) ===')
sample = grid_week[grid_week['fire_count'] > 0][feature_cols].head(20)
display(sample.round(2))

print(f'\nTotal rows in feature table:   {len(grid_week):,}')
print(f'Rows with fire_count > 0:      {(grid_week["fire_count"] > 0).sum():,}')
print(f'Rows with fire_count == 0:     {(grid_week["fire_count"] == 0).sum():,}')
print('\nNote: zero-fire rows teach the model what NOT-burning looks like (class imbalance is expected)')

---
## 10. Feature Correlation Analysis
Do our engineered features actually correlate with fire_count?  
If yes — our historical pattern assumption is validated.

In [ ]:
corr_cols = ['fire_count', 'fire_count_last_week', 'same_week_last_year',
             '3yr_avg', 'neighbor_fire_count', 'avg_frp',
             'avg_brightness', 'night_fire_pct', 'week_of_season']

# Use only rows with fire activity for more meaningful correlation
corr_data = grid_week[grid_week['fire_count'] > 0][corr_cols].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax,
            square=True, linewidths=0.5, annot_kws={'size': 9})

ax.set_title('Feature Correlation Matrix\n(rows with fire_count > 0)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('fig8_feature_correlation.png', bbox_inches='tight')
plt.show()

# Print top correlations with target
print('Correlations with fire_count (our target):')
print(corr_matrix['fire_count'].drop('fire_count').sort_values(ascending=False).round(3))

---
## 11. Final Dataset Summary

In [ ]:
print('=' * 55)
print('  FINAL DATASET SUMMARY')
print('=' * 55)
print(f'  Raw detections loaded:          {len(modis) + len(viirs_n20):>10,}')
print(f'  After confidence filter:        {len(fires):>10,}')
print(f'  After Punjab bbox filter:       {len(punjab) + (len(punjab)):>10,}')
print(f'  After Oct–Nov filter:           {len(punjab):>10,}')
print(f'  Unique 7km grid cells:          {punjab["grid_id"].nunique():>10,}')
print(f'  Years covered:                  {sorted(punjab["year"].unique())}')
print(f'  MODIS detections:               {(punjab["sensor"]=="MODIS").sum():>10,}')
print(f'  VIIRS detections:               {(punjab["sensor"]=="VIIRS").sum():>10,}')
print(f'')
print(f'  ML Feature Table rows:          {len(grid_week):>10,}')
print(f'  Active rows (fire_count > 0):   {(grid_week["fire_count"]>0).sum():>10,}')
print(f'  Number of features engineered:  {len(feature_cols)-3:>10}')
print('=' * 55)

# Save final feature table
grid_week[feature_cols].to_csv('punjab_feature_table_final.csv', index=False)
print('\n✅ Saved: punjab_feature_table_final.csv')
print('   → This CSV is the direct input to XGBoost in Phase 2')